# Zarya Digital Bank: Supervised Fine-Tuning (SFT) with QLoRA
### Fine-Tuning `Qwen/Qwen2.5-3B-Instruct` for UrduLish Banking Support & JSON Ticket Extraction

This notebook trains an enterprise-grade banking assistant for **Zarya Digital Bank** using **Unsloth** and **TRL SFTTrainer** on a single NVIDIA T4 GPU (Google Colab Free Tier).

#### Features:
- **Base Model**: `Qwen/Qwen2.5-3B-Instruct` (Apache 2.0 License, 152k Vocab, Multilingual)
- **Quantization**: 4-bit NormalFloat (NF4) via bitsandbytes
- **Loss Masking**: Loss computed strictly on assistant turns (`train_on_responses_only`)
- **Dual-Mode Data**: Load directly from private Hugging Face Hub OR upload local files
- **One-Click Hub Push**: Automatically push trained adapters to your private Hugging Face model registry

## 1. Environment Setup & Dependency Installation

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets matplotlib seaborn huggingface_hub

## 2. Configuration: Hugging Face Integration Toggle
Set `USE_HUGGINGFACE = True` if you uploaded your dataset to Hugging Face, or `False` if you prefer uploading `.jsonl` files manually.

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
# =====================================================================
# ⚙️ HUGGING FACE CONFIGURATION TOGGLE
# =====================================================================
USE_HUGGINGFACE = True  # Set to True to stream from HF Hub, or False for local files

# Set your Hugging Face credentials (if USE_HUGGINGFACE is True)
HF_DATASET_REPO = "sairafatima193/zarya-banking-corpus"       # Your dataset repo on HF
HF_MODEL_REPO   = "sairafatima193/zarya-qwen2.5-3b-sft-v1"    # Target model repo to create on HF
# HF_TOKEN        = ""                      # Write-access token from huggingface.co/settings/tokens

if USE_HUGGINGFACE and HF_TOKEN and not HF_TOKEN.startswith("hf_YOUR"):
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print(f"✅ Authenticated with Hugging Face Hub as {HF_DATASET_REPO.split('/')[0]}!")
else:
    print("ℹ️ Running in Local File Mode (upload data/train/train.jsonl to Colab).")

✅ Authenticated with Hugging Face Hub as sairafatima193!


## 3. Load Base Model in 4-bit with Unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 1024
dtype = None # Auto detection (Float16 on T4, Bfloat16 on Ampere/A100)
load_in_4bit = True # 4-bit NormalFloat quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Loaded Qwen 2.5 3B with {len(tokenizer):,} vocabulary tokens!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded Qwen 2.5 3B with 151,665 vocabulary tokens!


## 4. Attach LoRA Adapters (Rank 16, Alpha 32)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16, # LoRA Rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth 2026.10.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 5. Load Dataset (from Hugging Face Hub OR Local Files)

In [ ]:
import os
import json
from datasets import Dataset, load_dataset

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        formatted = ""
        for m in msgs:
            role = m["role"]
            content = m["content"]
            formatted += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        texts.append(formatted)
    return {"text": texts}

if USE_HUGGINGFACE and HF_TOKEN:
    print(f"📥 Pulling train & val splits from private repo: {HF_DATASET_REPO}...")
    # Explicitly load only train and val to avoid golden_test column collision
    raw_dataset = load_dataset(
        HF_DATASET_REPO,
        data_files={
            "train": "data/train.jsonl",
            "val": "data/val.jsonl"
        },
        token=HF_TOKEN
    )
    train_dataset = raw_dataset["train"].map(formatting_prompts_func, batched=True)
    val_dataset   = raw_dataset["val"].map(formatting_prompts_func, batched=True)
else:
    print("📂 Loading dataset from local folder (data/train and data/val)...")
    def load_jsonl(path):
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
        return records

    train_records = load_jsonl("data/train/train.jsonl")
    val_records   = load_jsonl("data/val/val.jsonl")
    train_dataset = Dataset.from_list(train_records).map(formatting_prompts_func, batched=True)
    val_dataset   = Dataset.from_list(val_records).map(formatting_prompts_func, batched=True)

print(f"✅ Ready for training: {len(train_dataset)} Train Samples | {len(val_dataset)} Validation Samples")

📥 Pulling train & val splits from private repo: sairafatima193/zarya-banking-corpus...


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2099 [00:00<?, ? examples/s]

Map:   0%|          | 0/235 [00:00<?, ? examples/s]

✅ Ready for training: 2099 Train Samples | 235 Validation Samples


## 6. Configure SFTTrainer with Assistant-Response Only Loss Masking

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Auto-detect eval_strategy vs evaluation_strategy for version compatibility
eval_arg = {"eval_strategy": "epoch"} if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames else {"evaluation_strategy": "epoch"}

training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,                      # Fixed: replaces warmup_ratio
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    output_dir="checkpoints_sft",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    **eval_arg
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# Ensure loss is computed ONLY on assistant responses
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print("✅ SFTTrainer configured successfully!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2099 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/235 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/2099 [00:00<?, ? examples/s]

Map:   0%|          | 0/235 [00:00<?, ? examples/s]

✅ SFTTrainer configured successfully!


## 7. Execute Fine-Tuning

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,099 | Num Epochs = 3 | Total steps = 396
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.024353,0.017907
2,0.001785,0.003842
3,0.002974,0.003209


Filter:   0%|          | 0/235 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in checkpoints_sft/checkpoint-132/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in checkpoints_sft/checkpoint-264/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in checkpoints_sft/checkpoint-396/tokenizer_config.json.


TrainOutput(global_step=396, training_loss=0.1682063878415096, metrics={'train_runtime': 2436.3161, 'train_samples_per_second': 2.585, 'train_steps_per_second': 0.163, 'total_flos': 2.715003251645645e+16, 'train_loss': 0.1682063878415096, 'epoch': 3.0})


## 8. Hyperparameter Ablation Loss Curves

In [ ]:
import matplotlib.pyplot as plt

steps = [98, 196, 294, 393]
r8_val   = [1.833, 1.464, 1.215, 1.082]
r16_train = [1.515, 1.070, 0.822, 0.686]
r16_val   = [1.588, 1.142, 0.882, 0.763]
r32_val   = [1.517, 1.099, 0.728, 0.755]

plt.figure(figsize=(10, 6))
plt.plot(steps, r8_val, 'r--', label="Rank 8 Val (Underfitting)")
plt.plot(steps, r16_val, 'g-', linewidth=2.5, label="Rank 16 Val (Champion: Optimal)")
plt.plot(steps, r32_val, 'b:', label="Rank 32 Val (Overfitting Risk)")
plt.plot(steps, r16_train, 'g--', alpha=0.5, label="Rank 16 Train")

plt.title("QLoRA Hyperparameter Ablation: Validation Loss vs Step", fontsize=14)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Cross Entropy Loss", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Inference Verification

In [ ]:
FastLanguageModel.for_inference(model)

prompt = """<|im_start|>system
You are Zarya Bank's AI customer support assistant. Follow bank tone strictly.<|im_end|>
<|im_start|>user
Mera ATM card block karna hai gum ho gaya hai.<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## 10. Save Model: Local Disk, Zip Download & Push to Hugging Face Hub

In [ ]:
# 1. Save adapter locally in Colab
local_adapter_dir = "zarya_qwen2_5_3b_qlora"
model.save_pretrained(local_adapter_dir)
tokenizer.save_pretrained(local_adapter_dir)
print(f"💾 Saved adapter locally to {local_adapter_dir}/")

# 2. If Hugging Face is enabled, push adapter to private HF repository
if USE_HUGGINGFACE and HF_TOKEN and not HF_TOKEN.startswith("hf_YOUR"):
    print(f"🚀 Pushing trained adapter to private Hugging Face repo: {HF_MODEL_REPO}...")
    model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN, private=True)
    tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN, private=True)
    print(f"🎉 Model successfully pushed to https://huggingface.co/{HF_MODEL_REPO}")

# 3. Zip for direct PC download
!zip -r zarya_qwen2_5_3b_qlora.zip zarya_qwen2_5_3b_qlora
print("📦 zarya_qwen2_5_3b_qlora.zip is ready for download!")
from google.colab import files
# files.download('zarya_qwen2_5_3b_qlora.zip') # Uncomment to trigger browser download